# 05.06 — Enum Properties: Coverage and Conditional Cardinality

> Builds on **01.05 — Conditional Cardinality**, **05.01 — Introducing the
> GraphProfile**, and **05.02 — Profile vs. Definition**.

A property is often a closed set of values: a `genre` is one of
`drama | action | comedy`, a `status` is `active | inactive`.  Python models
this with `enum.Enum`.  This notebook shows what orthograph does when an
**enum-typed** property meets the two places enums matter:

1. **Value coverage** (comparison) — does the observed data stay inside the
   declared enum?  Orthograph flags **both** directions, with deliberately
   different severities:
   - observed value **not in** the enum → `UNDECLARED_PROPERTY_VALUE` (**WARNING**) — the *data violates the contract*; this is the severe one.
   - declared value **never observed** → `UNOBSERVED_PROPERTY_VALUE` (**INFO**) — just an unused possibility; benign.
2. **Conditional cardinality** (validation) — using an enum-typed property as a
   discriminator in a `ConditionalCardinality` rule.

Along the way it documents three subtleties that were latent traps before this
work and are now handled:

- A graph DB stores an enum's **scalar** (a `String`/`Long`), never the Python
  enum object — so the structural type check must not raise a false
  `PROPERTY_TYPE_MISMATCH`.
- A **plain** `enum.Enum` member does not `==` its value, so a discriminator
  rule authored with the literal (`PropMatch({"genre": "drama"})`) must still
  match a node whose property is `Genre.DRAMA`.
- A **truncated** value distribution (`sample_complete=False`) can hide an
  undeclared value in its remainder, so the coverage check degrades to
  `PROPERTY_VALUE_UNVERIFIABLE` rather than a false all-clear.

Sections:
1. Declaring an enum-typed property
2. Value coverage — both directions flagged
3. The enum is stored as a scalar — no false type mismatch
4. Truncation honesty — an enum must be profiled completely
5. Enum as a conditional-cardinality discriminator
6. Plain enum vs. str-mixin enum — parity

In [10]:
import enum

from orthograph.compare import profile_to_definition
from orthograph.definition import (
    ConditionalCardinality,
    ConditionalRule,
    GraphDefinition,
    NodeModel,
    PropMatch,
    RelationshipModel,
    validate_data,
)
from orthograph.graph_profile.models import (
    BoundedDistribution,
    GraphProfile,
    NodeTypeProfile,
    PropertyProfile,
)

## 1. Declaring an enum-typed property

Declare `Genre` as a `str`-valued enum and use it as a `Movie` property.
`resolve_type_info` records the enum class as the property's `python_type`;
the comparison engine recognises it as an enum and switches on value-coverage
checking.

In [11]:
class Genre(str, enum.Enum):
    DRAMA = "drama"
    ACTION = "action"
    COMEDY = "comedy"


class Movie(NodeModel):
    __label__ = "Movie"
    __uid_field__ = "title"
    title: str
    genre: Genre


model = GraphDefinition(name="Films", node_types=[Movie], relationship_types=[])

spec = Movie.get_property_specs()["genre"]
print("genre python_type :", spec.python_type)
print("is Enum subclass  :", issubclass(spec.python_type, enum.Enum))
print("declared values   :", [m.value for m in Genre])

genre python_type : <enum 'Genre'>
is Enum subclass  : True
declared values   : ['drama', 'action', 'comedy']


## 2. Value coverage — both directions flagged

Build a profile where the observed `genre` values are `drama`, `action`, and
`romance`.  Two things are wrong relative to the declared enum:

- `romance` is observed but **not declared** → `UNDECLARED_PROPERTY_VALUE` (WARNING).
- `comedy` is declared but **never observed** → `UNOBSERVED_PROPERTY_VALUE` (INFO).

The histogram is **complete** (`sample_complete=True`), so both verdicts are
safe to make.

In [12]:
profile = GraphProfile(
    source="neo4j://prod:7687",
    node_type_profiles={
        "Movie": NodeTypeProfile(
            label="Movie",
            count=10,
            property_profiles={
                "title": PropertyProfile(
                    name="title",
                    present_count=10,
                    total_count=10,
                    observed_types=["String"],
                    constraint_required=True,
                ),
                "genre": PropertyProfile(
                    name="genre",
                    present_count=10,
                    total_count=10,
                    observed_types=["String"],  # the enum is stored as a String
                    constraint_required=True,
                    value_distribution=BoundedDistribution(
                        count=10,
                        histogram={"drama": 5, "action": 3, "romance": 2},
                        sample_complete=True,  # complete — both verdicts safe
                    ),
                ),
            },
        ),
    },
)

result = profile_to_definition(profile, model)

print("genre value-coverage findings:")
for issue in result.issues:
    if issue.entity_id == "Movie.genre" and "VALUE" in issue.code:
        print(f"  [{issue.severity.value.upper():7}] {issue.code}")
        print(f"            {issue.message}")

genre value-coverage findings:
  [WARNING] UNDECLARED_PROPERTY_VALUE
            Property 'genre' on Movie has observed value 'romance' not declared in enum 'Genre'
  [INFO   ] UNOBSERVED_PROPERTY_VALUE
            Property 'genre' on Movie declares enum value 'comedy' which was never observed


The severity split is intentional and matches ADR-034 §8:

| Direction | Code | Severity | Why |
|---|---|---|---|
| Observed ∉ declared | `UNDECLARED_PROPERTY_VALUE` | **WARNING** | The *data* breaks the declared contract — actionable. |
| Declared, never observed | `UNOBSERVED_PROPERTY_VALUE` | **INFO** | Just an unused possibility — informational. |

`UNDECLARED` is the one you act on: it means rows exist with a value your model
does not know about.

## 3. The enum is stored as a scalar — no false type mismatch

A graph database stores `Genre.DRAMA` as the string `"drama"`, so an inspector
reports the property's observed type as `String`, never `Genre`.  A naive
structural check (`observed String` vs `expected Genre`) would raise a spurious
`PROPERTY_TYPE_MISMATCH` for **every** enum property.

Orthograph compares the observed DB type against the enum's **value type**
(`str` for a string-valued enum, `int` for an int-valued one).  Confirm there is
no `PROPERTY_TYPE_MISMATCH` in the result above.

In [13]:
mismatches = [i for i in result.issues if i.code == "PROPERTY_TYPE_MISMATCH"]
print("PROPERTY_TYPE_MISMATCH count:", len(mismatches), " (expected 0)")
assert mismatches == [], "enum stored as its scalar must not mismatch"

# But a genuine wrong-scalar IS still caught: a string-valued enum stored as Long.
wrong_scalar = GraphProfile(
    source="x",
    node_type_profiles={
        "Movie": NodeTypeProfile(
            label="Movie",
            count=3,
            property_profiles={
                "title": PropertyProfile(
                    name="title",
                    present_count=3,
                    total_count=3,
                    observed_types=["String"],
                ),
                "genre": PropertyProfile(
                    name="genre",
                    present_count=3,
                    total_count=3,
                    observed_types=["Long"],  # stored as an int — genuinely wrong
                ),
            },
        ),
    },
)
bad = profile_to_definition(wrong_scalar, model)
wrong = [i for i in bad.issues if i.code == "PROPERTY_TYPE_MISMATCH"]
print("\nGenuine wrong-scalar (String-enum stored as Long):")
for i in wrong:
    print(f"  [{i.severity.value.upper()}] {i.code}: {i.message}")

PROPERTY_TYPE_MISMATCH count: 0  (expected 0)

Genuine wrong-scalar (String-enum stored as Long):
  [ERROR] PROPERTY_TYPE_MISMATCH: Property 'genre' on Movie has observed type 'Long' (Python: int), expected str


## 4. Truncation honesty — an enum must be profiled completely

A backend may cap a value distribution at the top-N values
(`sample_complete=False`), pushing the remainder into `other_count`.  For an
enum this is dangerous: **an undeclared value could be hiding in the remainder**
and the coverage check would silently miss it — exactly the breach the check
exists to catch.

Orthograph refuses to make a false all-clear.  When the histogram is truncated:

- a value **shown** in the histogram that is undeclared still raises
  `UNDECLARED_PROPERTY_VALUE` (it was definitely observed); but
- the `UNOBSERVED_PROPERTY_VALUE` verdict is **suppressed** (a declared value
  might be in the hidden remainder); and
- a single `PROPERTY_VALUE_UNVERIFIABLE` (INFO) is emitted to flag the gap.

First, the dangerous case: only the top value is kept and 40 observations are
hidden.

In [14]:
truncated_profile = GraphProfile(
    source="x",
    node_type_profiles={
        "Movie": NodeTypeProfile(
            label="Movie",
            count=100,
            property_profiles={
                "title": PropertyProfile(
                    name="title",
                    present_count=100,
                    total_count=100,
                    observed_types=["String"],
                ),
                "genre": PropertyProfile(
                    name="genre",
                    present_count=100,
                    total_count=100,
                    observed_types=["String"],
                    value_distribution=BoundedDistribution(
                        count=100,
                        histogram={"drama": 60},  # only top-1 kept
                        sample_complete=False,  # TRUNCATED
                        limit=1,
                        other_count=40,  # 40 hidden — could include undeclared values
                    ),
                ),
            },
        ),
    },
)

trunc_result = profile_to_definition(truncated_profile, model)
print("Truncated histogram (40 hidden) — genre findings:")
for i in trunc_result.issues:
    if i.entity_id == "Movie.genre" and "VALUE" in i.code:
        print(f"  [{i.severity.value.upper():7}] {i.code}")
        print(f"            {i.message}")

codes = {i.code for i in trunc_result.issues if i.entity_id == "Movie.genre"}
assert "PROPERTY_VALUE_UNVERIFIABLE" in codes, "must flag the truncation gap"
assert "UNOBSERVED_PROPERTY_VALUE" not in codes, "no false UNOBSERVED while truncated"
print("\nNo false UNOBSERVED verdict; the gap is honestly reported as UNVERIFIABLE.")

Truncated histogram (40 hidden) — genre findings:
  [INFO   ] PROPERTY_VALUE_UNVERIFIABLE
            Property 'genre' on Movie is declared as enum 'Genre' but its observed value distribution is truncated (40 observation(s) beyond the top-1 cap are hidden); undeclared values in the remainder cannot be detected and unobserved declared values cannot be confirmed

No false UNOBSERVED verdict; the gap is honestly reported as UNVERIFIABLE.


In [15]:
# Truncated AND a shown value is undeclared: the WARNING still fires for the
# shown value, plus the truncation UNVERIFIABLE for the hidden remainder.
truncated_with_undeclared = GraphProfile(
    source="x",
    node_type_profiles={
        "Movie": NodeTypeProfile(
            label="Movie",
            count=100,
            property_profiles={
                "genre": PropertyProfile(
                    name="genre",
                    present_count=100,
                    total_count=100,
                    observed_types=["String"],
                    value_distribution=BoundedDistribution(
                        count=100,
                        histogram={
                            "drama": 50,
                            "romance": 30,
                        },  # romance shown & undeclared
                        sample_complete=False,
                        limit=2,
                        other_count=20,
                    ),
                ),
            },
        ),
    },
)
res2 = profile_to_definition(truncated_with_undeclared, model)
print("Truncated + shown undeclared 'romance' — genre findings:")
for i in res2.issues:
    if i.entity_id == "Movie.genre" and "VALUE" in i.code:
        print(f"  [{i.severity.value.upper():7}] {i.code}: {i.message[:70]}...")

Truncated + shown undeclared 'romance' — genre findings:
  [WARNING] UNDECLARED_PROPERTY_VALUE: Property 'genre' on Movie has observed value 'romance' not declared in...
  [INFO   ] PROPERTY_VALUE_UNVERIFIABLE: Property 'genre' on Movie is declared as enum 'Genre' but its observed...


**Practical takeaway:** when a property is a declared enum, profile it with a
**complete** value distribution (no top-N cap).  The cardinality of an enum is
small by definition, so keeping every distinct value is cheap — and it is the
only way the comparison can give a definitive coverage verdict.

## 5. Enum as a conditional-cardinality discriminator

An enum-typed property can also drive a `ConditionalCardinality` rule
(see 01.05).  Here `Movie.genre` discriminates the source-side cardinality of
`ACTED_IN`:

| Person `kind` | Movie `genre` | Expected `ACTED_IN` (source) |
|---|---|---|
| `actor` | `drama` | 1..3 |
| `actor` | `blockbuster` | 0..1 |
| *(other)* | *(any)* | 0..* (default) |

The rule is authored with the **literal** value (`PropMatch({"genre": "drama"})`),
but a `Movie` instance's `genre` is an enum *member*.  Orthograph normalises the
observed member to its `.value` so the rule matches and the partition counts
are attributed correctly.

In [16]:
class CardGenre(str, enum.Enum):
    DRAMA = "drama"
    BLOCKBUSTER = "blockbuster"


class Person(NodeModel):
    __label__ = "Person"
    __uid_field__ = "name"
    name: str
    kind: str


class CardMovie(NodeModel):
    __label__ = "Movie"
    __uid_field__ = "title"
    title: str
    genre: CardGenre


acted_in = ConditionalCardinality(
    rules=(
        ConditionalRule(
            source=PropMatch({"kind": "actor"}),
            target=PropMatch({"genre": "drama"}),  # literal, not CardGenre.DRAMA
            spec="1..3",
        ),
        ConditionalRule(
            source=PropMatch({"kind": "actor"}),
            target=PropMatch({"genre": "blockbuster"}),
            spec="0..1",
        ),
    ),
    default="0..*",
)


class ActedIn(RelationshipModel):
    __label__ = "ACTED_IN"
    __source_label__ = "Person"
    __target_label__ = "Movie"
    __source_cardinality__ = acted_in
    __target_cardinality__ = "0..*"
    role: str


card_model = GraphDefinition(
    name="EnumFilmography",
    node_types=[Person, CardMovie],
    relationship_types=[ActedIn],
)
print("Conditional model with enum-typed genre discriminator assembled.")

Conditional model with enum-typed genre discriminator assembled.


In [17]:
# VALID: Alice (actor) — 2 drama roles (1..3 OK) + 1 blockbuster (0..1 OK).
nodes_ok = [
    Person(name="Alice", kind="actor"),
    CardMovie(title="D1", genre=CardGenre.DRAMA),
    CardMovie(title="D2", genre=CardGenre.DRAMA),
    CardMovie(title="B1", genre=CardGenre.BLOCKBUSTER),
]
rels_ok = [
    {
        "__label__": "ACTED_IN",
        "__source_uid__": "Alice",
        "__target_uid__": t,
        "role": "r",
    }
    for t in ("D1", "D2", "B1")
]
res_ok = validate_data(card_model, nodes=nodes_ok, relationships=rels_ok)
print(f"Valid case   — is_valid: {res_ok.is_valid}  errors: {len(res_ok.errors)}")

# VIOLATION: Bob (actor) — 5 drama roles breach 1..3.
nodes_bad = [Person(name="Bob", kind="actor")] + [
    CardMovie(title=f"X{i}", genre=CardGenre.DRAMA) for i in range(1, 6)
]
rels_bad = [
    {
        "__label__": "ACTED_IN",
        "__source_uid__": "Bob",
        "__target_uid__": f"X{i}",
        "role": "r",
    }
    for i in range(1, 6)
]
res_bad = validate_data(card_model, nodes=nodes_bad, relationships=rels_bad)
print(f"Violation case — is_valid: {res_bad.is_valid}")
for e in res_bad.errors:
    print(f"  [{e.code}] {e.message}")
    print(
        f"    partition target_kind={e.context['target_kind']!r}  actual={e.context['actual']}  expected={e.context['expected_min']}..{e.context['expected_max']}"
    )

Valid case   — is_valid: True  errors: 0
Violation case — is_valid: False
  [CARDINALITY_VIOLATION] Node 'Bob' (Person) has 5 outgoing ACTED_IN relationships for pair (source='actor', target='drama'), expected 1..3
    partition target_kind='drama'  actual=5  expected=1..3


Note the violation reports `target_kind='drama'` (the enum's underlying value),
the correct observed count `5`, and the resolved bound `1..3` — not a spurious
0-count or the raw `CardGenre.DRAMA` member.  The enum discriminator behaves
exactly as a plain string would.

## 6. Plain enum vs. str-mixin enum — parity

There are two ways to write a string-valued enum:

```python
class G1(enum.Enum):       # plain — G1.DRAMA != "drama"
    DRAMA = "drama"

class G2(str, enum.Enum):  # str-mixin — G2.DRAMA == "drama"
    DRAMA = "drama"
```

The **plain** form is the trap: `G1.DRAMA == "drama"` is `False`, so a literal
rule would never match without normalisation.  Orthograph normalises enum
members to their `.value` at the matching/partitioning boundary, so both forms
behave identically.  The cell below proves the equality difference and that the
validator result is the same for both.

In [18]:
class PlainGenre(enum.Enum):
    DRAMA = "drama"
    BLOCKBUSTER = "blockbuster"


print("Plain enum     PlainGenre.DRAMA == 'drama' :", PlainGenre.DRAMA == "drama")
print("str-mixin enum  CardGenre.DRAMA == 'drama' :", CardGenre.DRAMA == "drama")
print()


def build_and_validate(genre_enum):
    class P(NodeModel):
        __label__ = "Person"
        __uid_field__ = "name"
        name: str
        kind: str

    class M(NodeModel):
        __label__ = "Movie"
        __uid_field__ = "title"
        title: str
        genre: genre_enum  # type: ignore[valid-type]

    card = ConditionalCardinality(
        rules=(
            ConditionalRule(
                source=PropMatch({"kind": "actor"}),
                target=PropMatch({"genre": "drama"}),
                spec="1..3",
            ),
        ),
        default="0..*",
    )

    class AI(RelationshipModel):
        __label__ = "ACTED_IN"
        __source_label__ = "Person"
        __target_label__ = "Movie"
        __source_cardinality__ = card
        __target_cardinality__ = "0..*"
        role: str

    gd = GraphDefinition(name="G", node_types=[P, M], relationship_types=[AI])
    # Actor with 5 drama roles → breach of 1..3.
    nodes = [P(name="Z", kind="actor")] + [
        M(title=f"D{i}", genre=genre_enum.DRAMA) for i in range(1, 6)
    ]
    rels = [
        {
            "__label__": "ACTED_IN",
            "__source_uid__": "Z",
            "__target_uid__": f"D{i}",
            "role": "r",
        }
        for i in range(1, 6)
    ]
    return validate_data(gd, nodes, rels)


for name, ge in [("plain enum", PlainGenre), ("str-mixin enum", CardGenre)]:
    r = build_and_validate(ge)
    errs = [e for e in r.errors if e.code == "CARDINALITY_VIOLATION"]
    print(
        f"{name:15} — is_valid={r.is_valid}  violations={len(errs)}  "
        f"actual={errs[0].context['actual'] if errs else None}  "
        f"target_kind={errs[0].context['target_kind'] if errs else None!r}"
    )

Plain enum     PlainGenre.DRAMA == 'drama' : False
str-mixin enum  CardGenre.DRAMA == 'drama' : True

plain enum      — is_valid=False  violations=1  actual=5  target_kind='drama'
str-mixin enum  — is_valid=False  violations=1  actual=5  target_kind='drama'
